In [1]:
from pathlib import Path
from urllib.parse import urlparse
import re

import pandas as pd

# ----------------------------------
# Config
# ----------------------------------

# Root directory that contains all the cycling... folders
# If you're already "inside" that directory in the notebook, this can stay "."
ROOT = Path("survey_eye_tracker/eye_tracker_data")

# List of user session folders (relative to ROOT)
users = [
    "cycling932844b29e6175a85d195cbee96ce34057d0b2cb0b9bb90018e0301ef2460b82/2022_10_10_12_39_43", # 1
    "cycling132c5e4c5b0a45e274e7fb849fecd22e62edf409bd5f1b1322ddeb6d11f90d7d/2022_10_10_13_21_15", # 2
    "cycling0a3df224a10f3472c2a9c568a927406a49b012186f0983b9e10bcd883b4d5fcd/2022_10_18_14_08_36", # 3
    "cyclingbd1af6d2f4bda83c3d5d6dfc93817421d804a644ab12251d1033c885730217a4/2022_11_02_15_30_21", # 5
    "cycling28b744c8c0b8b330c7f678d5b23aa2ce614a5ae8143e96173fe3cde26ec2297e/2022_11_28_09_08_48", # 6
    "cycling145b3ad29cb766fb22e4cfba1d750db8c17470b8d07a6f01aad5918e20ccbe80/2022_11_29_09_51_39", # 7
    "cycling5876966995d4b61ed7073ec1ea1a92e1d3bcfbb02705d2bb441819922aaa89db/2022_11_29_10_23_22", # 8
    "cycling4eea1bbed89e15ea4b3ecbc10b941272711810e0f2648161ece9d5bcb9839dba/2022_11_29_10_56_24", # 9
    # "cycling34d1088782c31c6a960b5e97b47c878d06e548939499f39c90ed9e4209a128c3/2022_11_29_11_27_11", # 10 (excluded)
    "cycling8ffc01ebc87eb6aa9285e7688c79a4a6b63cf21a119820f13f054cd0e2fdd987/2022_11_29_13_54_31", # 11
    "cycling7e8315cff8453c95082b56e5b4745609cfda7bddd20bbc92c8f3f88dea3fd715/2022_12_01_09_09_42", # 12
    "cycling5e970a9dfb4a47cae2526d10a49e351fa97d26d6e24798cddf9e8ad77f6379fc/2022_12_01_09_52_46", # 13
    "cyclingd22a19aa45e85ca027d29be0fe3b839383d8566f1997284a96d2f97b8b5b9e63/2022_12_01_10_23_11", # 14
    "cycling684fdee4e2ba556e4e23a3f68062835cf9796cec92ffdbe9ce53171345f32e7b/2022_12_01_10_46_42", # 15
    # "cycling48a65889a9ec7a72c521c4c040cd9c8604a43cf428cd7ae3150d5518d505968c/2022_12_01_14_01_05", # 18 (excluded)
    "cycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd/2022_12_02_09_46_55", # 19
    "cyclingc08377f1e6826ffd8f74f4e1515d85319a2705749e0bb560f47e2e9c5c48186d/2022_12_02_10_18_23", # 20
    # "cyclingf7a6481ec3f7781563130f31e1bb429200913d03ea0dba8327f3a69132d978f1/2022_12_02_10_56_09", # 21 (excluded)
    "cyclingdbdde36bbe3344b160d31a87c5d85169c36650245f3d312494627ff1464bb2e4/2022_12_02_12_53_18", # 22
    "cycling5aa98a95dbd30e4ffd9d5f18d19cc095f093954581f2842e9daed80395793b90/2022_12_02_13_25_43", # 23
    "cyclingad8bc642880020eb31d2c1d5d10857bc864a01936bb335fe7a30e584ab21ebf8/2022_12_02_13_53_36", # 24
    "cycling469572b0c7fe5cc0c5f020ebae513ffeae62ec445e8b5c19154ba2e3ee1f6de4/2022_12_05_09_46_56", # 25
    "cycling61e4dc72e3a5c92061a3b8c78ea0f11334dcab587b2abebe99315c92213be055/2022_12_05_12_39_03", # 26
    "cycling4c845f8ebd5f514f1fdc690d2ab60d5f8beb818b464cea0fe96ca4f97a4f773e/2022_12_08_09_19_45", # 27
    "cycling2846319a17ec7fcad28ab540e7a7b18c9432e63357a06e9d15630eeb1ae62be3/2022_12_08_13_20_21", # 28
    "cycling14fd071ffaf930135bd748b9623a06847a49559438ae21c6cd8845252bae5462/2022_12_09_11_29_21", # 29
]

SCORES_FILENAME = "scores.csv"
OGAMA_FILENAME = "ogama_data.txt"
OUTPUT_PKL = "eyetracker_comparisons_df.pickle"


In [2]:
def normalize_path(p: str) -> str:
    """
    Normalize slashes and strip URL scheme if present.
    """
    if not isinstance(p, str):
        return ""
    s = str(p).strip().replace("\\", "/")
    if s.startswith("http://") or s.startswith("https://"):
        s = urlparse(s).path
    return s


def parse_dataset_and_stem(p: str):
    """
    From something like:
        "./images/berlin/3019.jpg"
        "./images/sequences/li_0_2.jpg"
    return:
        dataset = 'berlin' or 'sequences' (folder name)
        stem    = '3019', 'li_0_2', ...

    If cannot parse, return (None, None).
    """
    s = normalize_path(p)
    if not s:
        return None, None

    parts = s.split("/")
    if len(parts) < 2:
        return None, None

    filename = parts[-1]
    dataset = parts[-2].lower()
    stem = Path(filename).stem
    return dataset, stem


def make_npy_name(survey_id: int, trial_id: int, img_id: str, side: str) -> str:
    """
    side: 'left' or 'right'
    -> survey{survey_id}_trial{trial_id}_{img_id}_{side}_eyetrack.npy
    """
    return f"survey{survey_id}_trial{trial_id}_{img_id}_{side}_eyetrack.npy"


In [3]:
rows = []

print(f"ROOT directory: {ROOT.resolve()}")
print(f"Found {len(users)} user folders in the list.\n")

for survey_id, rel_path in enumerate(users, start=1):
    user_dir = ROOT / rel_path
    scores_path = user_dir / SCORES_FILENAME
    ogama_path = user_dir / OGAMA_FILENAME

    # 1) scores.csv is mandatory
    if not scores_path.exists():
        print(f"[Survey {survey_id}] {scores_path} NOT found → skipping this user.")
        continue

    # 2) has_eyetracker depends ONLY on ogama_data.txt existing
    has_eye_for_user = ogama_path.exists()
    print(
        f"[Survey {survey_id}] Using scores: {scores_path}, "
        f"ogama_data.txt present: {has_eye_for_user}"
    )

    try:
        df_scores = pd.read_csv(scores_path, header=None)
    except Exception as e:
        print(f"  ERROR reading {scores_path}: {e}")
        continue

    if df_scores.empty:
        print(f"  WARNING: {scores_path} is empty, skipping survey.")
        continue

    # Drop the *first* comparison (warm-up)
    if len(df_scores) <= 1:
        print(f"  WARNING: Only 1 row in {scores_path}, "
              "nothing left after dropping warm-up.")
        continue

    df_scores = df_scores.iloc[1:].reset_index(drop=True)

    # Rename for clarity: 0=timestamp, 1=subject, 2=left, 3=right, 4=score
    df_scores = df_scores.rename(columns={
        0: "timestamp",
        1: "subject",
        2: "left_path",
        3: "right_path",
        4: "score",
    })

    # Trial IDs: 1-based
    df_scores["trial_id"] = df_scores.index + 1

    # ------------------------------
    # Parse each comparison
    # ------------------------------
    for _, r in df_scores.iterrows():
        left_path = r["left_path"]
        right_path = r["right_path"]
        score_raw = r["score"]

        d_l, stem_l = parse_dataset_and_stem(left_path)
        d_r, stem_r = parse_dataset_and_stem(right_path)

        # Skip malformed paths
        if d_l is None or d_r is None or stem_l is None or stem_r is None:
            continue

        # Require both sides from the same dataset
        if d_l != d_r:
            continue

        # Convert score to int (-1, 0, 1)
        try:
            score = int(score_raw)
        except Exception:
            continue

        trial_id = int(r["trial_id"])

        # Rule 1 & 3: if ogama exists → build npy names, else None
        if has_eye_for_user:
            npy_file_l = make_npy_name(survey_id, trial_id, stem_l, "left")
            npy_file_r = make_npy_name(survey_id, trial_id, stem_r, "right")
            has_eyetracker = True
        else:
            npy_file_l = None
            npy_file_r = None
            has_eyetracker = False

        rows.append({
            "dataset": d_l,       # 'berlin', 'sequences', etc.
            "image_l": stem_l,
            "image_r": stem_r,
            "score": score,       # -1 / 0 / 1

            "has_eyetracker": has_eyetracker,
            "survey_id": survey_id,
            "trial_id": trial_id,
            "npy_file_l": npy_file_l,
            "npy_file_r": npy_file_r,
        })

# ------------------------
# Build DataFrame & save
# ------------------------
if not rows:
    print("\nWARNING: No usable eyetracker rows were found in any user folder.")
    eyetracker_df = pd.DataFrame()
else:
    eyetracker_df = pd.DataFrame(rows)

    # Drop exact duplicates across identifying info
    eyetracker_df = eyetracker_df.drop_duplicates(
        subset=[
            "dataset",
            "image_l",
            "image_r",
            "score",
            "survey_id",
            "trial_id",
            "npy_file_l",
            "npy_file_r",
        ]
    ).reset_index(drop=True)

    # Quick stats
    print("\n=== Eyetracker dataset summary ===")
    print("Total pairs (including scores-only users):", len(eyetracker_df))
    print("\nPairs per dataset:")
    print(eyetracker_df["dataset"].value_counts())

    print("\nScore distribution:")
    print(eyetracker_df["score"].value_counts().sort_index())

    print("\nExample rows:")
    display(eyetracker_df.head())

    # Save pickle
    eyetracker_df.to_pickle(OUTPUT_PKL)
    print(f"\nSaved eyetracker pickle to: {OUTPUT_PKL}")


ROOT directory: /home/csantiago/survey_eye_tracker/survey_eye_tracker/eye_tracker_data
Found 23 user folders in the list.

[Survey 1] survey_eye_tracker/eye_tracker_data/cycling932844b29e6175a85d195cbee96ce34057d0b2cb0b9bb90018e0301ef2460b82/2022_10_10_12_39_43/scores.csv NOT found → skipping this user.
[Survey 2] survey_eye_tracker/eye_tracker_data/cycling132c5e4c5b0a45e274e7fb849fecd22e62edf409bd5f1b1322ddeb6d11f90d7d/2022_10_10_13_21_15/scores.csv NOT found → skipping this user.
[Survey 3] survey_eye_tracker/eye_tracker_data/cycling0a3df224a10f3472c2a9c568a927406a49b012186f0983b9e10bcd883b4d5fcd/2022_10_18_14_08_36/scores.csv NOT found → skipping this user.
[Survey 4] survey_eye_tracker/eye_tracker_data/cyclingbd1af6d2f4bda83c3d5d6dfc93817421d804a644ab12251d1033c885730217a4/2022_11_02_15_30_21/scores.csv NOT found → skipping this user.
[Survey 5] survey_eye_tracker/eye_tracker_data/cycling28b744c8c0b8b330c7f678d5b23aa2ce614a5ae8143e96173fe3cde26ec2297e/2022_11_28_09_08_48/scores.csv